# Build the VOCAB Table

This notebook builds `vocab.csv` from the normalized token-level `corpus.csv`. Each row in the final table represents one unique term type (`term_str`) in the constitution corpus.

The table is tailored to this final project corpus and the columns requested in `FinalProject.ipynb`.

## Inputs and Outputs

**Inputs**
- `corpus.csv`: token-level constitution corpus with `term_str`, `pos`, and `pos_group`

**Output**
- `vocab.csv`: one row per unique term

The final `VOCAB` columns are:
- `term_str`
- `n`, `p`, `i`
- `df`, `idf`, `dfidf`
- `n_chars`
- `max_pos`, `max_pos_group`
- `n_pos`, `cat_pos`
- `n_pos_group`, `cat_pos_group`
- `stop`, `porter_stem`


## Notes on Measures

The table combines lexical frequency, document spread, and linguistic annotation.

- `n`: total number of times the term appears in the corpus
- `p`: relative frequency of the term in the corpus
- `i`: self-information, computed as `log2(1 / p)`
- `df`: document frequency, measured as the number of constitutions (`country_id`) in which the term appears
- `idf`: inverse document frequency, computed with SciKit-Learn-style smoothing as `log2((1 + N_docs) / (1 + df)) + 1`
- `dfidf`: a corpus-level significance score defined here as `df * idf`

This `dfidf` score is useful for surfacing terms with middling document frequency: terms that are neither too rare nor too common contribute the most, which matches the information-theoretic intuition behind self-entropy.

In [11]:
from pathlib import Path
import math

import nltk
import numpy as np
import pandas as pd
from nltk.stem import PorterStemmer

# Configure the core file paths and vocabulary-scoring options.
CORPUS_CSV = Path('corpus.csv')
OUTPUT_CSV = Path('vocab.csv')
IDF_METHOD = 'smooth'

# Domain-specific legal drafting terms treated as stopwords for this corpus.
CUSTOM_LEGAL_STOP_TERMS = {
    'article', 'articles', 'chapter', 'chapters', 'paragraph', 'paragraphs',
    'part', 'parts', 'provision', 'provisions', 'section', 'sections',
    'shall', 'subsection', 'subsections'
}

# Load stopwords from NLTK when available and fall back to a local list offline.
FALLBACK_STOPWORDS = {
    'a', 'an', 'and', 'are', 'as', 'at', 'be', 'been', 'but', 'by', 'for',
    'from', 'had', 'has', 'have', 'he', 'her', 'his', 'i', 'in', 'is', 'it',
    'its', 'of', 'on', 'or', 'that', 'the', 'their', 'them', 'they', 'this',
    'to', 'was', 'were', 'which', 'who', 'will', 'with', 'you', 'your'
}

try:
    from nltk.corpus import stopwords
    STOPWORDS = set(stopwords.words('english'))
    stopword_source = 'nltk.corpus.stopwords'
except LookupError:
    STOPWORDS = FALLBACK_STOPWORDS
    stopword_source = 'fallback stopword list'

STOPWORDS = STOPWORDS | CUSTOM_LEGAL_STOP_TERMS
stopword_source = f"{stopword_source} + custom legal stop terms"

PORTER = PorterStemmer()


## Load the Corpus

The vocabulary is derived from `corpus.csv`, not from the raw text files directly. This keeps the vocabulary aligned with the normalized corpus that already contains token normalization (`term_str`) and part-of-speech annotations (`pos`, `pos_group`). In the current pipeline, `term_str` keeps ordinary words and standard hyphenated forms, recovers simple leading enumeration patterns such as `3.Discounts` -> `discounts`, and drops noisier numeric/punctuation artifacts.

In [12]:
# Load the token-level fields needed to build the vocabulary table.
corpus = pd.read_csv(
    CORPUS_CSV,
    usecols=['country_id', 'term_str', 'pos', 'pos_group'],
)

# Standardize blanks before filtering out empty normalized terms.
corpus['term_str'] = corpus['term_str'].fillna('').astype(str).str.strip()
corpus = corpus[corpus['term_str'] != ''].copy()

print(f'Corpus rows used for VOCAB: {len(corpus):,}')
print(f'Unique constitutions: {corpus["country_id"].nunique()}')
print(f'Unique terms: {corpus["term_str"].nunique():,}')
print(f'Stopword source: {stopword_source}')
print(f'Custom legal stop terms: {len(CUSTOM_LEGAL_STOP_TERMS)}')
corpus.head()


Corpus rows used for VOCAB: 3,839,736
Unique constitutions: 192
Unique terms: 24,735
Stopword source: fallback stopword list + custom legal stop terms
Custom legal stop terms: 15


,country_id,pos,term_str,pos_group
0,Afghanistan,IN,in,IN
1,Afghanistan,DT,the,DT
2,Afghanistan,NN,name,NN
3,Afghanistan,IN,of,IN
4,Afghanistan,NNP,allah,NN


## Build the VOCAB Table

This step aggregates the token-level corpus to one row per `term_str`. It computes frequency columns first, then adds document-frequency measures, then derives POS summaries and lexical annotations such as stopword status and a Porter stem. The `stop` flag combines a general English stopword list with a small set of constitution-specific legal drafting terms so downstream models can exclude low-information boilerplate consistently.

In [13]:
def sorted_set_string(values) -> str:
    # Store unique values in a stable string form for easier inspection.
    items = sorted({str(v) for v in values if pd.notna(v) and str(v) != ''})
    return '{' + ', '.join(items) + '}' if items else '{}'


def dominant_value(series: pd.Series) -> str:
    # Break ties deterministically so repeated runs keep the same label.
    counts = series.value_counts()
    if counts.empty:
        return ''
    top_count = counts.iloc[0]
    tied = sorted(counts[counts == top_count].index.astype(str).tolist())
    return tied[0]


def build_vocab(corpus_df: pd.DataFrame) -> pd.DataFrame:
    # Compute frequency, document-frequency, and POS summary features per term.
    total_terms = len(corpus_df)
    n_docs = corpus_df['country_id'].nunique()

    # Start from corpus term counts so each row in VOCAB represents one term type.
    vocab = corpus_df['term_str'].value_counts().rename('n').to_frame()
    vocab.index.name = 'term_str'
    vocab['n_chars'] = vocab.index.str.len()
    vocab['p'] = vocab['n'] / total_terms
    vocab['i'] = np.log2(1 / vocab['p'])

    # Count in how many constitutions each term appears.
    df = corpus_df.groupby('term_str')['country_id'].nunique().rename('df')
    vocab = vocab.join(df)
    if IDF_METHOD == 'smooth':
        # Use smoothed IDF so terms that appear everywhere still get finite weights.
        vocab['idf'] = np.log2((1 + n_docs) / (1 + vocab['df'])) + 1
    else:
        vocab['idf'] = np.log2(n_docs / vocab['df'])
    vocab['dfidf'] = vocab['df'] * vocab['idf']

    # Summarize the dominant and full sets of POS labels for each term.
    pos_stats = corpus_df.groupby('term_str').agg(
        max_pos=('pos', dominant_value),
        max_pos_group=('pos_group', dominant_value),
        n_pos=('pos', 'nunique'),
        n_pos_group=('pos_group', 'nunique'),
        cat_pos=('pos', sorted_set_string),
        cat_pos_group=('pos_group', sorted_set_string),
    )
    vocab = vocab.join(pos_stats)

    # Add lexical flags used in later filtering and interpretation.
    vocab['stop'] = vocab.index.to_series().isin(STOPWORDS).astype(int)
    vocab['porter_stem'] = vocab.index.to_series().apply(PORTER.stem)

    vocab = vocab.reset_index()
    # Reorder columns to match the exported project schema.
    vocab = vocab[[
        'term_str', 'n', 'p', 'i', 'df', 'idf', 'dfidf', 'n_chars',
        'max_pos', 'max_pos_group', 'n_pos', 'cat_pos',
        'n_pos_group', 'cat_pos_group', 'stop', 'porter_stem'
    ]]
    return vocab.sort_values(['dfidf', 'n', 'term_str'], ascending=[False, False, True]).reset_index(drop=True)


VOCAB = build_vocab(corpus)
print(f'VOCAB rows: {len(VOCAB):,}')
VOCAB.head(20)


VOCAB rows: 24,735


,term_str,n,p,i,df,idf,dfidf,n_chars,max_pos,max_pos_group,n_pos,cat_pos,n_pos_group,cat_pos_group,stop,porter_stem
0,relating,1787,0.000465,11.069252,142,1.432586,203.427169,8,VBG,VB,4,"{NN, NNP, NNS, VBG}",2,"{NN, VB}",0,relat
1,procedures,933,0.000243,12.006842,142,1.432586,203.427169,10,NNS,NN,9,"{IN, JJ, NN, NNP, NNS, POS, VB, VBP, VBZ}",5,"{IN, JJ, NN, POS, VB}",0,procedur
2,committees,861,0.000224,12.122706,142,1.432586,203.427169,10,NNS,NN,8,"{JJ, NN, NNP, NNS, VB, VBD, VBP, VBZ}",3,"{JJ, NN, VB}",0,committe
3,regulation,769,0.000200,12.285736,142,1.432586,203.427169,10,NN,NN,7,"{CD, JJ, NN, NNP, VB, VBP, VBZ}",4,"{CD, JJ, NN, VB}",0,regul
4,ten,741,0.000193,12.339246,142,1.432586,203.427169,3,JJ,JJ,13,"{CD, JJ, JJR, NN, NNP, NNPS, NNS, RB, VB, VBD,...",5,"{CD, JJ, NN, RB, VB}",0,ten
5,thirty,741,0.000193,12.339246,142,1.432586,203.427169,6,JJ,JJ,9,"{CC, CD, IN, JJ, NN, NNP, VB, VBP, VBZ}",6,"{CC, CD, IN, JJ, NN, VB}",0,thirti
6,professional,698,0.000182,12.425492,142,1.432586,203.427169,12,JJ,JJ,7,"{CD, JJ, NN, NNP, VB, VBN, VBP}",4,"{CD, JJ, NN, VB}",0,profession
7,last,692,0.000180,12.437947,142,1.432586,203.427169,4,JJ,JJ,7,"{CD, FW, JJ, NN, VB, VBP, VBZ}",5,"{CD, FW, JJ, NN, VB}",0,last
8,called,653,0.000170,12.521637,142,1.432586,203.427169,6,VBN,VB,8,"{JJ, NN, NNS, RB, VBD, VBN, VBP, VBZ}",4,"{JJ, NN, RB, VB}",0,call
9,institution,598,0.000156,12.648574,142,1.432586,203.427169,11,NN,NN,10,"{CD, IN, JJ, NN, NNP, NNS, VB, VBD, VBP, VBZ}",5,"{CD, IN, JJ, NN, VB}",0,institut


## Top 20 Terms by DFIDF

These terms are the current top 20 by the `dfidf` score defined above. This is the same list you can report in the `VOCAB` section of `FinalProject.ipynb`.

In [14]:
top20 = VOCAB[['term_str', 'dfidf', 'df', 'idf', 'n']].head(20)
top20


,term_str,dfidf,df,idf,n
0,relating,203.427169,142,1.432586,1787
1,procedures,203.427169,142,1.432586,933
2,committees,203.427169,142,1.432586,861
3,regulation,203.427169,142,1.432586,769
4,ten,203.427169,142,1.432586,741
5,thirty,203.427169,142,1.432586,741
6,professional,203.427169,142,1.432586,698
7,last,203.427169,142,1.432586,692
8,called,203.427169,142,1.432586,653
9,institution,203.427169,142,1.432586,598


## Save `vocab.csv`

This final cell writes the completed vocabulary table to disk for later project steps such as bag-of-words construction, feature filtering, TFIDF building, and downstream modeling notebooks.


In [15]:
# Save the completed VOCAB table for downstream use.
VOCAB.to_csv(OUTPUT_CSV, index=False)
print(f'Saved {len(VOCAB):,} VOCAB rows to: {OUTPUT_CSV.resolve()}')
print('Columns:', ', '.join(VOCAB.columns))


Saved 24,735 VOCAB rows to: C:\Users\garre\school\spring_2026\ds_5001\vocab.csv
Columns: term_str, n, p, i, df, idf, dfidf, n_chars, max_pos, max_pos_group, n_pos, cat_pos, n_pos_group, cat_pos_group, stop, porter_stem
